### Lab 3.2 Backpropagation

In this lab you will inspect the gradients in a neural network and understand how they are computed as they propagate from the loss backwards through the network.

In [15]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

Let's make some random data: a 2D input point and a label (set to one).

In [16]:
x = torch.randn(2)
y = torch.ones(1).long()

In [17]:
x, y

(tensor([0.1653, 0.0823]), tensor([1]))

Now let's make the parameters needed for a multi-layer perceptron with a single hidden layer of three neurons.  We will be sure to set `requires_grad=True` so that PyTorch knows to compute the gradients for these tensors.

In [18]:
W1 = torch.randn(3,2,requires_grad=True)
b1 = torch.randn(3,requires_grad=True)

W2 = torch.randn(1,3,requires_grad=True)
b2 = torch.randn(1,requires_grad=True)

Now we compute the score output and a squared error loss term.  This is the "forward" step as explained in the backpropagation notes.

In [19]:
h = W1@x+ b1
s = F.relu(h)

z = W2@s + b2

L = 0.5*(z-y)**2

`retain_grad()` tells PyTorch not to throw away the gradients of intermediate tensors.

In [20]:
h.retain_grad()
s.retain_grad()
z.retain_grad()
L.retain_grad()

Now we call `backward()` to compute the gradients.

In [21]:
L.backward()

Let's think about what the gradient of the loss w.r.t. $z$ should be.
$$L = \frac{1}{2}(z-y)^2$$
$$\frac{dL}{dz} = (z-y)$$

In [22]:
dLdz = z-y

Let's check our answer with PyTorch's answer.

In [23]:
dLdz, z.grad

(tensor([-2.7635], grad_fn=<SubBackward0>), tensor([-2.7635]))

Yup, they're the same!

Now, we work backward to obtain the gradients for $W^2$ and $\vec{b}^2$ as explained in the backprogation notes.

In [24]:
dLdW2 = dLdz[:,None] @ s[None,:]

In [25]:
dLdW2, W2.grad

(tensor([[-0.0430, -4.2318,  0.0000]], grad_fn=<MmBackward0>),
 tensor([[-0.0430, -4.2318, -0.0000]]))

In [26]:
dzdb2 = torch.eye(1)

In [27]:
dLdb2 = dLdz @ dzdb2

Note that it would have been more efficient to simply do `dLdb2 = dLdz` since multiplying by $1$ has no effect.

In [28]:
dLdb2, b2.grad

(tensor([-2.7635], grad_fn=<SqueezeBackward4>), tensor([-2.7635]))

### Exercises

Continue to work back until you can calculate the derivatives w.r.t. $W_1$ and $\vec{b_1}$.

1. Calculate $dL/d\vec{s}$ and check your answer.


2. Compute $dL/d\vec{h}$ and check your answer.

3. Compute $dL/dW^1$ and $dL/d\vec{b^1}$ and check your answers.